# Late Interaction using ColBERT
ColBERT is a late interaction technique to improve search quality.
<br>
In **Late Interaction** using ColBERT :
* embeddings are created for every token in the search query and the document.
* for each query token we find a document token with maximum cosine similarity.
* these maximum similarity (`MaxSim`) scores are aggregated to produce the final document relevance score.

> **NOTE**<br> An honest ColBERT implementation requires one vector per token of each document and MaxSim scoring server-side.<br>
However, turbopuffer doesn't support multi-vector documents or server-side MaxSim ranking today.

Since turbopuffer does not natively support it, a practical ColBERT implementation today is a two-stage approximation:
* server-side dense ANN retrieval followed by 
* client-side MaxSim re-ranking on the top-k candidates.

Key implementation decisions:
* At index time, ColBERT is run on Every document and token vectors are stored as a JSON attribute in each document row.
* At query time:    
    * turbopuffer ANN finds top-100 dense vectors fast
    * turbopuffer returns token_vectors attribute for those 100 docs in the same response (no additional round trip)
    * client computes MaxSim on the pre-computed token matrices.

Advantages of this solution:
* Pre-computed tokens once at index-time, makes every query fast. 
* Single roundtrip to turbopuffer.
* Token vectors travel with the document, no lookups or joins.
* Storing JSON blobs is relatively cheaper in turbopuffer's storage model.


### Step 1: Setup
Imports, load .env, initialize turbopuffer client

In [7]:
import os
import json
import numpy as np
import turbopuffer
from dotenv import load_dotenv

# Load variables from .env
load_dotenv()

# Access variables
TURBOPUFFER_API_KEY = os.getenv("TURBOPUFFER_API_KEY")

#initialize turbopuffer client
tpuf = turbopuffer.Turbopuffer(
    # API tokens are created in the dashboard: https://turbopuffer.com/dashboard
    api_key=os.getenv("TURBOPUFFER_API_KEY"),
    # Pick the right region: https://turbopuffer.com/docs/regions
    region="gcp-us-central1",
)

#initialize a namespace
ns = tpuf.namespace(f'colbert-late-interaction-int8')

### Step 2: Dataset
Load [Quora Question Paris](https://huggingface.co/datasets/sentence-transformers/quora-duplicates) from HuggingFace

In [2]:
import pandas as pd

df_qpair = pd.read_parquet("hf://datasets/sentence-transformers/quora-duplicates/pair/train-00000-of-00001.parquet")
df_qpair.count()

anchor      149263
positive    149263
dtype: int64

In [3]:
df_qpair.head(100)

,anchor,positive
0,Astrology: I am a Capricorn Sun Cap moon and c...,"I'm a triple Capricorn (Sun, Moon and ascendan..."
1,How can I be a good geologist?,What should I do to be a great geologist?
2,How do I read and find my YouTube comments?,How can I see all my Youtube comments?
3,What can make Physics easy to learn?,How can you make physics easy to learn?
4,What was your first sexual experience like?,What was your first sexual experience?
...,...,...
95,Will Modi win in 2019?,Can Narendra Modi become Prime Minister of Ind...
96,"What exactly is the ""Common Core Initiative/St...",What are the pros and cons of the Common Core ...
97,How do I choose a journal to publish my paper?,Where do I publish my paper?
98,What are your New Year's resolutions for 2017?,What is your creative New Year's resolution fo...


The dataset has ~150K questions pairs as `anchor` and `positive`columns. <br>We will:
* take a subset of 1000 questions pairs
* use `positive` to build the corpus, create dense embeddings + `token_vector` attribute in turbopuffer
* use a few `anchor` questions to compare the search performance of **dense retrieval** vs **late interaction**

Succes metric would be "did the matching positive for given anchor rank #1".

In [4]:
df_corpus = df_qpair['positive'][:1000]
print(f"Dataframe shape: {df_corpus.shape}")
print(f"Corpus length is {df_corpus.count()} \nCorpus Sample:")
df_corpus.head()

Dataframe shape: (1000,)
Corpus length is 1000 
Corpus Sample:


0    I'm a triple Capricorn (Sun, Moon and ascendan...
1            What should I do to be a great geologist?
2               How can I see all my Youtube comments?
3              How can you make physics easy to learn?
4               What was your first sexual experience?
Name: positive, dtype: str

### Step 3: Embedding
Load ColBERT model via fastembed, generate dense vector and token vectors per document.<br>
> We choose [`fastembed`](https://huggingface.co/ferrisS/harrier-oss-v1-270m-fastembed) because it is a lightweight embedding library with native late interaction support, and runs on CPU.

In [5]:
from fastembed import TextEmbedding, LateInteractionTextEmbedding

dense_model = TextEmbedding("sentence-transformers/all-MiniLM-L6-v2")
colbert_model = LateInteractionTextEmbedding("colbert-ir/colbertv2.0")

#### Token Vector Quantization (int8)

ColBERT token vectors are float32 values in the range -1 to 1. Storing them as 
JSON strings is expensive - each float like `0.0234567` takes ~10 characters.

Quantizing to int8 maps the float range (-1 to 1) into integer range (-127 to 127):

```python
token_vectors = (colbert_embedding * 127).astype(np.int8).tolist()
```

- Multiply by 127 - scales -1.0 → -127 and 1.0 → 127
- Cast to int8 - integers like `-12` or `43` are far shorter strings than floats
- At query time, divide by 127 to convert back to approximate float32 before MaxSim

This loses a small amount of precision but reduces payload size by ~82%.

In [8]:
# Loop over all 1000 corpus questions, generate dense vectors and token vectors.

records = []

for i in range(len(df_corpus)):
    text = df_corpus[i]
    
    # dense vector from MiniLM - for ANN retrieval
    dense_vector = list(dense_model.embed([text]))[0].tolist()
    
    # token vectors from ColBERT - for MaxSim reranking, quantized to int8
    colbert_embedding = list(colbert_model.embed([text]))[0]
    token_vectors = (colbert_embedding * 127).astype(np.int8).tolist()
    
    record = {
        "id": i + 1,
        "vector": dense_vector,
        "token_vectors": json.dumps(token_vectors),
        "text": text
    }
    
    records.append(record)

print(f"Embedded {len(records)} documents")
print(f"Dense vector dims: {len(records[0]['vector'])}")

Embedded 1000 documents
Dense vector dims: 384


### Step 4: Index
Upsert to turbopuffer:
* `dense_vector` column for ANN, 
* `token_vectors` as JSON attribute, 
* `text` as attribute
> **Note** `token_vectors` is a non-filterable attribute,, so paying for a filterable index on them would be wasteful. <br>
 Set `filterable: false` in the schema for a 50% storage cost discount.

In [9]:
# Upsert documents with vectors and attributes
# idempotent - safe to re-run, existing rows will be overwritten not duplicated

ns.write(
    upsert_rows=records,
    distance_metric='cosine_distance',
    schema={
        "token_vectors": {"type": "string", "filterable": False},
        "text": {"type": "string", "filterable": False}
    }
)

print(f"Upserted {len(records)} documents to turbopuffer")

Upserted 1000 documents to turbopuffer


### Step 5: Query: Dense only
Run a test query, time it, show top 5 results

In [10]:
import time

def dense_search(query, top_k=5):
    query_vector = list(dense_model.embed([query]))[0].tolist()

    start_ts = time.perf_counter()
    results = ns.query(
        rank_by=["vector", "ANN", query_vector],
        limit=top_k,
        include_attributes=['text', 'token_vectors']
    )
    end_ts = time.perf_counter()
    latency = (end_ts - start_ts) * 1000

    return results.rows, latency, results.billing

# dense search test
results, latency, billing = dense_search(df_qpair["anchor"][1]) #use an anchor question from the original question 

print(f"Original question pair: {df_qpair.iloc[1]}")
print(f"Latency: {latency:.1f}ms")
for row in results:
    print(f"Distance: {row['$dist']:.4f} | {row.text}")
    
print(f"Bytes queried: {billing.billable_logical_bytes_queried / 1024 / 1024:.2f} MB")

Original question pair: anchor                 How can I be a good geologist?
positive    What should I do to be a great geologist?
Name: 1, dtype: str
Latency: 226.0ms
Distance: 0.0630 | What should I do to be a great geologist?
Distance: 0.5984 | How do I become a good computer science engineer?
Distance: 0.6042 | How do you become a good teacher?
Distance: 0.6219 | How can you make physics easy to learn?
Distance: 0.6279 | What are some good mountaineering institutes for a beginner level mountaineering course?
Bytes queried: 244.14 MB


### Step 6: Query: Late Interaction
Same query, ANN top-100, client-side MaxSim re-rank, time it, show top 5 results

In [13]:
import numpy as np

#MaxSim function
def maxsim(query_tokens, doc_tokens):
    query_tokens = np.array(query_tokens) #(Q, 128) matrix
    doc_tokens = np.array(doc_tokens) #(D, 128) matrix
    scores = np.dot(query_tokens, doc_tokens.T) #score is (Q, D) matrix
    max_similarities = scores.max(axis=1) #max similarity score for each query token against the whole document.
    maxim_score = max_similarities.sum() #final aggregated maxim socre of the query against the document

    return maxim_score 

#Full ColBERT search function
def colbert_search(query, top_k=5):
    query_embedding = list(colbert_model.embed([query]))[0]

    #stage 1 - dense retrieval of top 100 ANN documents
    candidates, dense_latency, dense_billing = dense_search(query, top_k=100)

    #stage 2 - rerank with MaxSim
    start_ts = time.perf_counter()
    scores = []
    for candidate in candidates:
        doc_tokens = np.array(json.loads(candidate.token_vectors)) / 127 #de-quantization step
        score = maxsim(query_embedding, doc_tokens)
        scores.append((score, candidate.text))

    scores.sort(key=lambda x: x[0], reverse=True)
    end_ts = time.perf_counter()
    colbert_latency = (end_ts - start_ts) * 1000

    return scores[:top_k], dense_latency, colbert_latency, dense_billing

# colbert search test
results, dense_latency, colbert_latency, dense_billing = colbert_search(df_qpair["anchor"][1])

print(f"Original question pair: {df_qpair.iloc[1]}")
print(f"\nColBERT total latency: {dense_latency + colbert_latency:.1f}ms (dense: {dense_latency:.1f}ms + rerank: {colbert_latency:.1f}ms)\n")
for score, text in results:
     print(f"Score: {score:.4f} | {text}")

print(f"Bytes queried: {dense_billing.billable_logical_bytes_queried / 1024 / 1024:.2f} MB")


Original question pair: anchor                 How can I be a good geologist?
positive    What should I do to be a great geologist?
Name: 1, dtype: str

ColBERT total latency: 428.1ms (dense: 402.1ms + rerank: 26.0ms)

Score: 8.1361 | What should I do to be a great geologist?
Score: 6.3656 | What should I do to be a good Speaker?
Score: 6.1502 | How do you become a good teacher?
Score: 5.9140 | What is to be done to be a good software developer?
Score: 5.6304 | How should I be a people person?
Bytes queried: 244.14 MB


### Step 7: Evaluation
#### Recall@K Evaluation

To quantitatively compare dense retrieval and ColBERT late interaction, we use **Recall@K**, the fraction of queries where the known correct answer appears in the top K results.

The Quora duplicates dataset gives us ground truth: each `anchor` question has a known `positive` duplicate in the corpus. A perfect retrieval system should return the duplicate as rank #1.

We evaluate across 25 queries at two thresholds:

- **Recall@1** — did the correct answer come back as rank #1? Measures ranking precision.
- **Recall@5** — did the correct answer appear anywhere in the top 5? Measures retrieval coverage.


In [14]:
def evaluate(queries_df, top_k=5, n=25):
    dense_hits = 0
    colbert_hits = 0
    
    for i in range(n):
        anchor = queries_df.iloc[i]['anchor']
        positive = queries_df.iloc[i]['positive']
        
        # dense search
        dense_results, _, _ = dense_search(anchor, top_k=top_k)
        dense_texts = [row.text for row in dense_results]
        if positive in dense_texts:
            dense_hits += 1
        
        # colbert search
        colbert_results, _, _, _ = colbert_search(anchor, top_k=top_k)
        colbert_texts = [text for _, text in colbert_results]
        if positive in colbert_texts:
            colbert_hits += 1
    
    print(f"Dense Recall@{top_k}:   {dense_hits/n:.2f} ({dense_hits}/{n})")
    print(f"ColBERT Recall@{top_k}: {colbert_hits/n:.2f} ({colbert_hits}/{n})")

evaluate(df_qpair)

Dense Recall@5:   1.00 (25/25)
ColBERT Recall@5: 1.00 (25/25)


In [15]:
#Recall@1 over 50 samples
evaluate(df_qpair, top_k=1, n=50)

Dense Recall@1:   0.94 (47/50)
ColBERT Recall@1: 0.84 (42/50)


In [ ]:
# Eval with custom queries with slightly more complexity than in the original corpus 
custom_queries = [
    "techniques for improving oxygen flow through nasal passages during sleep",
    "methods to track viewer engagement on uploaded video content",
    "strategies for generating income from online video platforms"
]

for query in custom_queries:
    print(f"\nQuery: {query}")
    print("-" * 60)
    
    # dense results
    dense_results, _, _ = dense_search(query, top_k=3)
    print(f"Dense ({dense_latency:.1f}ms):")
    for row in dense_results:
        print(f"  {row['$dist']:.4f} | {row.text}")
    
    # colbert results
    colbert_results, _, _, _ = colbert_search(query, top_k=3)
    print(f"ColBERT rerank ({colbert_latency:.1f}ms):")
    for score, text in colbert_results:
        print(f"  {score:.4f} | {text}")


Query: techniques for improving oxygen flow through nasal passages during sleep
------------------------------------------------------------
Dense (438.9ms):
  0.3364 | How can I keep my nose from getting stuffy at night?
  0.5080 | How can I efficiently learn while sleeping?
  0.6078 | How do you apply Vicks Vapor rub to treat a stuffy nose?
ColBERT rerank (28.7ms):
  5.1410 | How can I efficiently learn while sleeping?
  4.2629 | What are marketing techniques for language schools?
  4.1311 | How can I keep my nose from getting stuffy at night?

Query: methods to track viewer engagement on uploaded video content
------------------------------------------------------------
Dense (438.9ms):
  0.4390 | How can I see who viewed my video I just posted on Instagram?
  0.6598 | What are the best apps for downloading films for free?
  0.6810 | How do you upload movies on YouTube and monetize them? Is there any issue of copyright
ColBERT rerank (28.7ms):
  4.4263 | How can I see all my Youtub

In [16]:
#Latency evaluation
# warm up
dense_search("How do I make money through YouTube?")

# measure warm
results, dense_latency, billing = dense_search("How do I make money through YouTube?")
print(f"Dense warm latency: {dense_latency:.1f}ms")

scored, dense_l, rerank_l, billing = colbert_search("How do I make money through YouTube?")
print(f"ColBERT warm latency: {dense_l + rerank_l:.1f}ms (dense: {dense_l:.1f}ms + rerank: {rerank_l:.1f}ms)")

Dense warm latency: 89.3ms
ColBERT warm latency: 214.8ms (dense: 187.8ms + rerank: 27.1ms)


## Analysis: Dense Retrieval vs ColBERT Late Interaction

### Setup
- **Corpus:** 1,000 questions from the Quora Duplicates dataset
- **Dense model:** `sentence-transformers/all-MiniLM-L6-v2` (384 dimensions)
- **ColBERT model:** `colbert-ir/colbertv2.0` (128 dimensions per token)
- **Evaluation:** Does the known duplicate question appear in the top K results?

---

### Result quality
> Dense retrieval won on this dataset. ColBERT did not improve quality here.<br>
> The dataset is probably too simple (single-sentence paraphrases) to show ColBERT's strengths.

| Metric | Dense (MiniLM) | ColBERT Late Interaction |
|---|---|---|
| Recall@1 (n=50) | 0.94 (47/50) | 0.84 (42/50) |
| Recall@5 (n=25) | 1.00 (25/25) | 1.00 (25/25) |

I used the Quora Duplicates dataset because each question has a known duplicate, this gives me a ground truth to measure against. For each test query I checked whether the correct duplicate appeared in the top results.

After digging into why, it makes sense. The Quora dataset is mostly paraphrases, questions that mean the same thing but use slightly different words. Dense retrieval is already very good at this. 
ColBERT's token-level matching can get confused, for instance the word "sleep" in a query strongly matched "sleeping" in a completely unrelated document, pushing the correct result down the ranking.

The results are also ambiguous partly because here each document is only one sentence long. ColBERT was designed for passage retrieval. <br>
Had the document been more complex, with multiple competing ideas, ColBERT probably would have outshined dense retrieval.

---

### Latency
> Late interaction is 4-5x slower on warm queries. The bottleneck is 
> fetching token vectors for 100 candidates, not the MaxSim math itself.

| Approach | turbopuffer Query | MaxSim Rerank | Total |
|---|---|---|---|
| Dense only (top_k=10) | 70ms | 0ms | 70ms |
| ColBERT float32 JSON (top_k=100) | 275ms | 50ms | 325ms |
| ColBERT int8 JSON (top_k=100) | 188ms | 27ms | 214ms |

I measured latency on a warm cache, meaning turbopuffer had already loaded the namespace into its NVMe SSD from a previous query.

The surprising part: the slow step is not the MaxSim math (only 50ms). The slow step is fetching 100 documents worth of token vectors back from turbopuffer. Each document has ~26 tokens × 128 floating point numbers stored as a JSON string. Returning 100 of those per query adds up fast.

Quantizing token vectors to int8 before storing reduces payload size by 82% — from 72KB to 12.8KB per document. This translates to a 34% end-to-end latency improvement (325ms → 214ms) with only two lines of code change. The MaxSim computation also gets faster because int8 matrix operations are cheaper than float32.

One thing to note is that the first query to a cold namespace is much slower (I saw up to 990ms) because turbopuffer has to load data from S3 into its NVMe cache first.

---

### Cost
> Query cost is identical. Storage is 10x larger but still cheap — 
> only ~$0.36/month extra at 1M documents. Latency is the real tradeoff, not cost.

| | Dense Only | ColBERT float32 JSON | ColBERT int8 JSON |
|---|---|---|---|
| turbopuffer query cost | baseline | same | same |
| Storage (1k docs) | ~2MB estimated | 19.63MB actual | ~3.5MB estimated |
| Extra compute per query | none | MaxSim on 100 candidates | MaxSim on 100 candidates |

In this experiment I ran two namespaces — float32 and int8 — totalling 2,000 documents and 25.62MB storage on the turbopuffer dashboard. The int8 namespace is roughly 3.5MB vs 19.63MB for float32 — an 82% storage reduction consistent with the payload size experiment.

The query cost is the same for all three approaches — turbopuffer scans the same namespace either way. 190GB queried across 746 queries in this experiment, at turbopuffer's pricing that's a few cents total.

Scaling storage to 1M documents:
- Dense only: ~2GB → ~$0.04/month
- ColBERT float32: ~20GB → ~$0.40/month
- ColBERT int8: ~3.5GB → ~$0.07/month

int8 quantization brings storage cost almost in line with dense only — a $0.03/month difference at 1M documents. The latency overhead remains the more meaningful tradeoff, not cost. MaxSim calculations happen entirely client-side, so VM compute costs should also be factored in for production deployments.

---

### Bottom Line
> Don't use late interaction by default. Measure on your own dataset first. 
> It shines on long, multi-concept documents, not short paraphrases.

| Use Case | Recommendation |
|---|---|
| Questions, paraphrases, short text | Dense retrieval is simpler and faster |
| Technical docs, code, long text | Try late interaction, worth the overhead |
| Need sub-100ms warm latency | Dense only |
| OK with ~600ms warm latency | Late interaction viable |
| Storage cost | Object storage is relatively cheap |

Late interaction might not always be better than dense retrieval. It adds complexity, latency, and storage overhead. Whether those tradeoffs are worth depends entirely on your data and your query patterns. 

> **NOTE** These recommendations are based on the 2-stage approach used in this guide, where turbopuffer only supports dense retrieval via ANN and ColBERT's MaxSim is actually calculated client-side, for now.

## D2: Roadmap: How turbopuffer Could Better Support Late Interaction

Building this guide exposed two places where turbopuffer could make late 
interaction significantly easier and faster to implement.

---

### 1. Native Multi-Vector Column Support

Right now I had to store token vectors as a JSON string attribute, it works 
but it's slow to fetch and deserialize at query time. This added ~470ms to 
every ColBERT query in my testing.

If turbopuffer supported a native multi-vector column type, token matrices 
could be stored in compact binary format instead of JSON strings. This would 
be the single biggest latency improvement for late interaction on turbopuffer today.

```python
# what this could look like
schema={
    "token_vectors": {
        "type": "multi_vector",
        "dimensions": 128,
        "filterable": False
    }
}
```

---

### 2. Server-Side MaxSim Scoring

Right now MaxSim reranking happens in my Python code after fetching results 
from turbopuffer. It works but it means pulling 100 documents worth of token 
vectors over the network on every query.

Server-side MaxSim would be the natural equivalent for multi-vector columns.
Colocate compute and data, return only the final ranked results.

```python
# what this could look like
results = ns.query(
    rank_by=["token_vectors", "MaxSim", query_token_vectors],
    limit=10
)
```

---

### Why This Matters

Weaviate already supports native multi-vector and server-side MaxSim. 
turbopuffer's object storage architecture makes late interaction dramatically cheaper, 
native multi-vector support would make it faster too.